# Pauli Propagation: Theory Tutorial

<img src="./assets/pictures.svg" width="600">

This notebook introduces **Pauli propagation**, the core algorithm behind `pprop`. We work through a small example by hand and then verify every step with the library.

## Background

Computing the expectation value of an observable $O$ under a parametrized circuit $U(\boldsymbol{\theta})$ starting from $|0\rangle$ amounts to evaluating

$$\langle O \rangle(\boldsymbol{\theta}) = \langle 0 | U^\dagger(\boldsymbol{\theta})\, O\, U(\boldsymbol{\theta}) | 0 \rangle.$$

The standard approach simulates the statevector $U(\boldsymbol{\theta})|0\rangle$, which costs $O(2^n)$ memory. Pauli propagation takes the opposite route: it evolves $O$ **backwards** through the circuit in the Heisenberg picture,

$$O \;\to\; U^\dagger\, O\, U,$$

keeping track of the result as a **sum of Pauli words** with trigonometric polynomial coefficients. At the end, only the Pauli words made of $I$ or $Z$ contribute to the final expectation value.

In the $|0\rangle^{\otimes n}$ computational basis state, only Pauli words composed entirely of $Z$ and $I$ operators have non-zero expectation:

$$
        \langle 0 | I | 0 \rangle = 1, \quad
        \langle 0 | Z | 0 \rangle = 1, \quad
        \langle 0 | X | 0 \rangle = 0, \quad
        \langle 0 | Y | 0 \rangle = 0
$$

## 1. Propagation rules

Each gate $G(\theta)$ defines a **conjugation map** on Pauli operators $P \mapsto G^\dagger(\theta)\, P\, G(\theta)$. For the two gates in our example:

**RY gate** ($RY(\theta) = e^{-i\theta Y/2}$):

$$X \to \cos\theta\, X + \sin\theta\, Z$$
$$Z \to \cos\theta\, Z - \sin\theta\, X$$

**CNOT gate** (control on qubit 1, target on qubit 0):

$$Z \otimes I \to Z \otimes Z$$
$$X \otimes I \to X \otimes I$$

All other single-qubit Paulis on qubits not in the gate support are left unchanged. The rules above are derived directly from the conjugation relations and hold exactly for any parameter value.

## 2. The circuit

We propagate the observable $Z_0$ (Pauli Z on qubit 0) backwards through the following two-qubit circuit:

```
q0: RY(theta_0) -- [target of CNOT] -- RY(theta_1) -- measure Z
q1:              -- [control of CNOT]
```

Note the gate order for propagation is **reversed**: we start from the observable and apply gates from right to left, so the sequence is $RY(\theta_1)$, then CNOT, then $RY(\theta_0)$.

In [ ]:
from pprop import Propagator
import pennylane as qml

def ansatz(params):
    qml.RY(params[0], wires=0)
    qml.CNOT([1, 0])
    qml.RY(params[1], wires=0)
    return qml.expval(qml.Z(0))

prop = Propagator(ansatz)
prop.show()

## 3. Propagation by hand

We track the evolving Pauli word step by step. Gates are applied in reverse circuit order.

**Starting observable:** $Z_0 \otimes I_1$

---

**Step 1:** Apply $RY(\theta_1) \otimes I$ using the rule $Z \to \cos\theta Z - \sin\theta X$:

$$Z_0 \otimes I_1 \;\to\; \cos(\theta_1)\, Z_0 \otimes I_1 \;-\; \sin(\theta_1)\, X_0 \otimes I_1$$

---

**Step 2:** Apply CNOT(1, 0) using the rule $Z_0 \otimes I_1 \to Z_0 \otimes Z_1$ (X on the target is unchanged):

$$\cos(\theta_1)\, Z_0 \otimes I_1 - \sin(\theta_1)\, X_0 \otimes I_1
\;\to\;
\cos(\theta_1)\, Z_0 \otimes Z_1 - \sin(\theta_1)\, X_0 \otimes I_1$$

---

**Step 3:** Apply $RY(\theta_0) \otimes I$ using $Z \to \cos\theta Z - \sin\theta X$ and $X \to \cos\theta X + \sin\theta Z$:

$$\cos(\theta_1)\, Z_0 \otimes Z_1 - \sin(\theta_1)\, X_0 \otimes I_1$$

$$\to\; +\cos(\theta_0)\cos(\theta_1)\, Z_0 \otimes Z_1
\;-\; \sin(\theta_0)\cos(\theta_1)\, X_0 \otimes Z_1$$

$$\;-\; \cos(\theta_0)\sin(\theta_1)\, X_0 \otimes I_1
\;-\; \sin(\theta_0)\sin(\theta_1)\, Z_0 \otimes I_1$$

---

**Step 4:** $\langle 0|\cdot|0\rangle$ is non-zero only for the identity on every qubit. Among the four terms above, only $Z_0 \otimes I_1$ contributes (since $\langle 0|Z|0\rangle = 1$ and $\langle 0|I|0\rangle = 1$, while $\langle 0|X|0\rangle = 0$). Therefore:

$$\langle Z_0 \rangle(\boldsymbol{\theta}) = -\sin(\theta_0)\sin(\theta_1) + \cos(\theta_0)\cos(\theta_1)$$

## 4. The closed-form expression

`prop.expression(0)` returns the symbolic trigonometric polynomial for the first (and only) observable. It should match the hand-derived result.

In [ ]:
prop.propagate()
prop.expression(0)

## 5. Numerical evaluation

We can now evaluate $\langle Z_0 \rangle$ at any parameter point in microseconds, with no statevector simulation. We verify against the analytical formula.

In [ ]:
import numpy as np

rng    = np.random.default_rng(0)
params = rng.uniform(0, 2 * np.pi, size=prop.num_params)

pprop_val    = prop(params)[0]
analytic_val = -np.sin(params[0])*np.sin(params[1]) + np.cos(params[0])*np.cos(params[1])

print(f"pprop value    : {pprop_val:.10f}")
print(f"Analytic value : {analytic_val:.10f}")
print(f"Difference     : {abs(pprop_val - analytic_val):.2e}")

## 6. Exact gradients

Because the expression is a trigonometric polynomial, its gradient with respect to $\boldsymbol{\theta}$ is also exact. `eval_and_grad` returns both the value and the gradient vector in a single call.

The analytical gradient are:

$$\frac{\partial}{\partial \theta_0} = -\cos(\theta_0)\sin(\theta_1) - \sin(\theta_0)\cos(\theta_1), \qquad \frac{\partial}{\partial \theta_1} = -\sin(\theta_0)\cos(\theta_1) - \cos(\theta_0)\sin(\theta_1)$$

In [ ]:
vals, grads = prop.eval_and_grad(params)

pprop_grad    = np.array(grads[0])
analytic_grad = np.array([
    -np.cos(params[0])*np.sin(params[1]) - np.sin(params[0])*np.cos(params[1]),
    -np.sin(params[0])*np.cos(params[1]) - np.cos(params[0])*np.sin(params[1]),
])

print(f"pprop gradient    : {pprop_grad}")
print(f"Analytic gradient : {analytic_grad}")
print(f"Max difference    : {np.max(np.abs(pprop_grad - analytic_grad)):.2e}")